# CNN Benchmark: 8 Architectures on MNIST (custom NumPy NN library)

Clones the library from GitHub, imports the `nn` classes **directly into the notebook** (no subprocess hacks), runs gradient checks + smoke tests + the full benchmark, and produces all results/analysis: summary tables, latency benchmarks (inference ms, throughput, train-step), loss/accuracy curves, accuracy-vs-params/time scatter, Pareto plot, confusion matrices, sample predictions, and architecture diagrams.

**Runtime:** Runtime → Change runtime type → **CPU** (the library is pure NumPy — a GPU adds nothing).

Rough timing on free Colab CPU: sanity cells ≈ 3 min; full benchmark (5 epochs × 8 models) ≈ 1.5–2.5 h. Start with the subset settings in cell 3.

In [ ]:
#@title 1. Clone the repo
REPO_URL = "https://github.com/dhakalnirajan/internship-projects.git"  #@param {type:"string"}
BRANCH = "main"                     #@param {type:"string"}

import os

# clone (or pull if already cloned)
if os.path.isdir('/content/repo/.git'):
    !git -C /content/repo pull
else:
    !git clone --depth 1 --branch "$BRANCH" "$REPO_URL" /content/repo

%cd /content/repo
!ls

In [ ]:
#@title 2. Import the nn library classes into the notebook
import sys
sys.path.insert(0, '/content/repo')   # make `import nn` / `import cnn_benchmark` work

# --- core library ---
import numpy as np
import matplotlib.pyplot as plt

from nn.autograd import Tensor
from nn.autograd.ops import concat
from nn.activations import relu, sigmoid, tanh, softmax
from nn.layers import (Dense, Conv2D, Conv1D, MaxPooling2D, AveragePooling2D,
                       Flatten, Dropout, Activation)
from nn.losses import CrossEntropy, MSE
from nn.optimizers import Adam, SGD
from nn.models import Sequential

# --- benchmark package (architectures + harness + gradient checks) ---
from cnn_benchmark.architectures import build_models
from cnn_benchmark.harness import evaluate, to_onehot, train_model
from cnn_benchmark.run_benchmark import load_mnist
from cnn_benchmark import grad_check

print('nn library imported OK')
x = Tensor(np.random.rand(2, 28, 28, 1).astype(np.float32), requires_grad=True)
c = Conv2D(4, 3, padding='same')
y = relu(c(x))
print('forward/backward sanity:', y.shape, y.requires_grad)

In [ ]:
#@title 3. Settings
SUBSET_TRAIN = 20000      #@param {type:"integer"}   # 60000 = full; start with 20000
EPOCHS = 5                #@param {type:"integer"}
BATCH_SIZE = 64           #@param {type:"integer"}
LEARNING_RATE = 0.001     #@param {type:"number"}
print(f"subset={SUBSET_TRAIN}, epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}")

In [ ]:
#@title 4. Get MNIST
import os, urllib.request
if not os.path.exists('/content/repo/mnist.npz'):
    print('downloading mnist.npz ...')
    urllib.request.urlretrieve('https://s3.amazonaws.com/img-datasets/mnist.npz', '/content/repo/mnist.npz')
print('mnist.npz ready:', os.path.getsize('/content/repo/mnist.npz') // 1048576, 'MB')

In [ ]:
#@title 5. Gradient checks (vectorized conv & pooling vs numerical differentiation)
grad_check.check_conv2d()
grad_check.check_pooling()
print('\nrel_err < ~1e-2 means the analytic gradients are correct')

In [ ]:
#@title 6. Smoke test: 1 forward+backward step on every architecture (~2 min)
rng = np.random.default_rng(42)
X = rng.random((64, 28, 28, 1)).astype(np.float32)
y_oh = to_onehot(rng.integers(0, 10, size=64))

for m in build_models():
    try:
        h = train_model(m, X, y_oh, epochs=1, batch_size=16, max_batches=2, verbose=False)
        status = f"OK   loss={h['loss'][0]:.4f}"
    except Exception as e:
        status = f"FAIL {type(e).__name__}: {e}"
    print(f"{m.name:<18} params={m.count_params():>9,}  {status}")

In [ ]:
#@title 7. Full benchmark (the long cell — respects the settings from cell 3)
import time, json

X_train, y_train, X_val, y_val, X_test, y_test = load_mnist()
X_train, y_train = X_train[:SUBSET_TRAIN], y_train[:SUBSET_TRAIN]
y_train_oh, y_val_oh = to_onehot(y_train), to_onehot(y_val)

results = []
trained_models = {}   # name -> trained model object, used by cells 13/14
for model in build_models():
    print(f"\n=== {model.name} ===")
    t0 = time.time()
    model.forward(X_train[:2], training=True)          # lazy-build layers
    n_params = model.count_params()
    history = train_model(model, X_train, y_train_oh,
                          X_val=X_val[:2000], y_val=y_val[:2000],
                          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE)
    train_s = time.time() - t0
    test_acc = evaluate(model, X_test, y_test)
    results.append({"model": model.name, "params": n_params,
                    "train_s": round(train_s, 1),
                    "final_loss": history["loss"][-1],
                    "val_acc": history["val_acc"][-1],
                    "test_acc": test_acc, "history": history})
    trained_models[model.name] = model
    print(f"  params={n_params:,}  test_acc={test_acc:.4f}  ({train_s:.0f}s)")

with open('cnn_benchmark/results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
print('\nSaved cnn_benchmark/results.json')

In [ ]:
#@title 7b. Latency benchmark — inference latency, throughput, train-step (+ charts)
import time, json

def _lat_stats(fn, warmup=5, runs=30):
    for _ in range(warmup):
        fn()
    ts = np.empty(runs)
    for i in range(runs):
        t0 = time.perf_counter()
        fn()
        ts[i] = (time.perf_counter() - t0) * 1000.0
    return float(ts.mean()), float(np.median(ts)), float(np.percentile(ts, 95))

xb1, xb64, xb256 = X_test[:1], X_test[:64], X_test[:256]
y64_oh = to_onehot(y_test[:64])
loss_fn = CrossEntropy()

for r in results:
    m = trained_models[r['model']]

    # single-image inference latency (what matters when serving)
    mean_ms, med_ms, p95_ms = _lat_stats(lambda: m.forward(xb1, training=False))

    # batched throughput at batch=256
    t0 = time.perf_counter()
    for _ in range(5):
        m.forward(xb256, training=False)
    batch_ms = (time.perf_counter() - t0) / 5.0 * 1000.0
    imgs_s = 256.0 / (batch_ms / 1000.0)

    # training step latency: forward + backward, batch=64
    def step():
        pred = m.forward(xb64, training=True)
        loss = loss_fn.forward(y64_oh, pred)
        loss.backward()
    step_ms, _, _ = _lat_stats(step, warmup=2, runs=10)

    r['lat_ms'] = round(mean_ms, 3)
    r['p95_ms'] = round(p95_ms, 3)
    r['imgs_s'] = round(imgs_s, 1)
    r['step_ms'] = round(step_ms, 2)
    print(f"{r['model']:<18} infer={mean_ms:.2f}ms (p95 {p95_ms:.2f})  "
          f"throughput={imgs_s:.0f} img/s  train_step={step_ms:.1f}ms")

# persist so the table in cell 9 and later cells see the latency numbers
with open('cnn_benchmark/results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
by_lat = sorted(results, key=lambda r: r['lat_ms'])
names = [r['model'] for r in by_lat]

axes[0].barh(names, [r['lat_ms'] for r in by_lat], color='tab:red')
axes[0].set_title('Inference latency per image (ms, lower is better)')
for i, r in enumerate(by_lat):
    axes[0].text(r['lat_ms'], i, f" {r['lat_ms']:.2f}", va='center', fontsize=9)

axes[1].barh(names, [r['imgs_s'] for r in by_lat], color='tab:cyan')
axes[1].set_title('Throughput @ batch 256 (images/s, higher is better)')

for r in results:
    axes[2].scatter(r['lat_ms'], r['test_acc'], s=90)
    axes[2].annotate(r['model'], (r['lat_ms'], r['test_acc']),
                     textcoords='offset points', xytext=(6, 5), fontsize=9)
axes[2].set_xlabel('latency per image (ms)')
axes[2].set_ylabel('test accuracy')
axes[2].set_title('Accuracy vs latency (bottom-left = best)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('charts_latency.png', dpi=150)
plt.show()

In [ ]:
#@title 8. Architecture diagrams (layer-by-layer shape table per model)
import pandas as pd

def describe_layer(layer):
    """Short human-readable description of a layer."""
    t = type(layer).__name__
    if t == 'Conv2D':
        return f"Conv2D {layer.filters} filters, {layer.kernel_size}, stride={layer.strides}, pad={layer.padding}"
    if t == 'Dense':
        act = layer.activation.__name__ if layer.activation else 'linear'
        return f"Dense {layer.units} ({act})"
    if t == 'MaxPooling2D' or t == 'AveragePooling2D':
        return f"{t} {layer.pool_size} stride={layer.strides}"
    if t == 'Dropout':
        return f"Dropout p={layer.rate}"
    if t == 'Activation':
        return f"Activation({layer.activation.__name__})"
    if t == 'Flatten':
        return 'Flatten'
    if callable(layer) and not hasattr(layer, 'forward'):
        return f"Activation({layer.__name__})"   # bare relu/softmax functions
    return t

def trace_shapes(model, input_shape=(1, 28, 28, 1)):
    """Run one forward pass recording output shape after each layer."""
    x = np.zeros(input_shape, dtype=np.float32)
    rows = []
    if hasattr(model, 'seq'):                          # SequentialModel
        layers = model.seq.layers
        h = x
        for L in layers:
            h_data = h.data if hasattr(h, 'data') else h
            t = Tensor(h_data, requires_grad=False)
            if hasattr(L, 'training'):
                L.training = False
            if hasattr(L, 'forward') and 'training' in L.forward.__code__.co_varnames:
                h = L(t, training=False)
            else:
                h = L(t)
            rows.append((describe_layer(L), tuple(h.data.shape)))
    else:                                              # ResNet/GoogLeNet/etc — single I/O
        out = model.forward(Tensor(x), training=False)
        rows.append((f"{model.name} (multi-branch module)", tuple(out.data.shape)))
    return rows

models = {m.name: m for m in build_models()}
for name, m in models.items():
    print(f"\n{'='*70}\n{name}  ({m.count_params():,} params)\n{'='*70}")
    rows = trace_shapes(m)
    if rows:
        df = pd.DataFrame(rows, columns=['Layer', 'Output shape'])
        print(df.to_string(index=False))

In [ ]:
#@title 9. Results — summary table (sorted by test accuracy)
results = json.load(open('cnn_benchmark/results.json'))
results.sort(key=lambda r: -r['test_acc'])

cols = ('model', 'params', 'final_loss', 'val_acc', 'test_acc', 'train_s',
        'lat_ms', 'p95_ms', 'imgs_s', 'step_ms')
summary = pd.DataFrame([{**{k: r.get(k) for k in cols},
                         'params(M)': round(r['params'] / 1e6, 3)}
                        for r in results])
summary.index = range(1, len(summary) + 1)
summary.index.name = 'rank'
summary

In [ ]:
#@title 10. Charts — bars: accuracy / params / time + convergence curves
names = [r['model'] for r in results]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].barh(names[::-1], [r['test_acc'] for r in results][::-1], color='tab:blue')
axes[0].set_title('Test accuracy'); axes[0].set_xlim(0.0, 1.0)
for i, r in enumerate(results[::-1]):
    axes[0].text(r['test_acc'] + 0.01, i, f"{r['test_acc']:.3f}", va='center', fontsize=9)

axes[1].barh(names[::-1], [r['params'] for r in results][::-1], color='tab:orange')
axes[1].set_title('Parameters')

axes[2].barh(names[::-1], [r['train_s'] for r in results][::-1], color='tab:green')
axes[2].set_title('Train time (s)')

markers = cycle = iter(['o', 's', '^', 'v', 'D', 'P', 'X', '*'])
for r in results:
    va = r['history']['val_acc']
    axes[3].plot(range(1, len(va) + 1), va, marker=next(markers), label=r['model'])
axes[3].set_title('Validation accuracy per epoch'); axes[3].set_xlabel('epoch')
axes[3].set_ylabel('val accuracy'); axes[3].legend(fontsize=8); axes[3].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('charts_bars.png', dpi=150); plt.show()

In [ ]:
#@title 11. Charts — training loss curves (all models on one axis)
plt.figure(figsize=(9, 5))
for r in results:
    L = r['history']['loss']
    plt.plot(range(1, len(L) + 1), L, marker='o', label=r['model'])
plt.title('Training loss per epoch')
plt.xlabel('epoch'); plt.ylabel('cross-entropy loss')
plt.yscale('log')          # losses span decades; log scale keeps curves readable
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('charts_loss_curves.png', dpi=150); plt.show()

In [ ]:
#@title 12. Charts — accuracy vs params vs time (Pareto view)
# Answers: which model buys the most accuracy per parameter / per second?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in results:
    axes[0].scatter(r['params'] / 1e3, r['test_acc'], s=90)
    axes[0].annotate(r['model'], (r['params'] / 1e3, r['test_acc']),
                     textcoords='offset points', xytext=(6, 5), fontsize=9)
axes[0].set_xscale('log')
axes[0].set_xlabel('parameters (thousands, log scale)'); axes[0].set_ylabel('test accuracy')
axes[0].set_title('Accuracy vs model size'); axes[0].grid(alpha=0.3)

for r in results:
    axes[1].scatter(r['train_s'], r['test_acc'], s=90)
    axes[1].annotate(r['model'], (r['train_s'], r['test_acc']),
                     textcoords='offset points', xytext=(6, 5), fontsize=9)
axes[1].set_xlabel('training time (s)'); axes[1].set_ylabel('test accuracy')
axes[1].set_title('Accuracy vs training time'); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('charts_pareto.png', dpi=150); plt.show()

# efficiency leaderboard: accuracy gained per second of training
eff = sorted(results, key=lambda r: -r['test_acc'] / max(r['train_s'], 1))
print('\nEfficiency (test_acc per train-second):')
for r in eff:
    print(f"  {r['model']:<18} {r['test_acc'] / max(r['train_s'], 1):.5f}  (acc={r['test_acc']:.3f}, {r['train_s']:.0f}s)")

In [ ]:
#@title 13. Confusion matrices (top-4 models) + per-class accuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

top = results[:4]
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, r in zip(axes.ravel(), top):
    m = trained_models[r['model']]
    preds = []
    for s in range(0, X_test.shape[0], 512):
        p = m.forward(X_test[s:s+512], training=False)
        preds.append(np.argmax(p.data, axis=1))
    preds = np.concatenate(preds)
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['model']} (acc={r['test_acc']:.3f})")
plt.tight_layout(); plt.savefig('charts_confusion.png', dpi=150); plt.show()

# per-class accuracy of the best model
best = trained_models[results[0]['model']]
preds = np.concatenate([np.argmax(best.forward(X_test[s:s+512], training=False).data, axis=1)
                        for s in range(0, X_test.shape[0], 512)])
per_class = [np.mean(preds[y_test == c] == c) for c in range(10)]
plt.figure(figsize=(8, 4))
plt.bar(range(10), per_class, color='tab:purple')
plt.ylim(0.8, 1.0); plt.xticks(range(10))
plt.title(f"Per-class accuracy — {results[0]['model']}")
plt.xlabel('digit'); plt.ylabel('accuracy')
plt.tight_layout(); plt.savefig('charts_per_class.png', dpi=150); plt.show()

In [ ]:
#@title 14. Sample predictions gallery (best model)
n_show = 12
idx = rng.choice(X_test.shape[0], n_show, replace=False)
p = best.forward(X_test[idx], training=False)
pred = np.argmax(p.data, axis=1)

plt.figure(figsize=(13, 4.5))
for i, j in enumerate(idx):
    plt.subplot(2, 6, i + 1)
    plt.imshow(X_test[j, :, :, 0], cmap='gray')
    ok = pred[i] == y_test[j]
    plt.title(f"pred {pred[i]} / true {y_test[j]}", color='green' if ok else 'red', fontsize=10)
    plt.axis('off')
plt.suptitle(f"Sample predictions — {results[0]['model']} (green=correct, red=wrong)")
plt.tight_layout(); plt.savefig('charts_samples.png', dpi=150); plt.show()

# show a few mistakes too
wrong = np.where(preds != y_test)[0][:6]
if len(wrong):
    plt.figure(figsize=(11, 2.5))
    for i, j in enumerate(wrong):
        plt.subplot(1, 6, i + 1)
        plt.imshow(X_test[j, :, :, 0], cmap='gray')
        plt.title(f"pred {preds[j]} / true {y_test[j]}", color='red', fontsize=10)
        plt.axis('off')
    plt.suptitle('Misclassified examples (best model)')
    plt.tight_layout(); plt.savefig('charts_mistakes.png', dpi=150); plt.show()

In [ ]:
#@title 15. Export — download all charts + results.json
from google.colab import files

artifacts = ['cnn_benchmark/results.json', 'charts_bars.png', 'charts_loss_curves.png', 'charts_latency.png',
             'charts_pareto.png', 'charts_confusion.png', 'charts_per_class.png',
             'charts_samples.png', 'charts_mistakes.png']
for a in artifacts:
    if os.path.exists(a):
        files.download(a)
print('downloaded:', len(artifacts), 'artifacts')

## Notes

- **Pretrained weights?** ImageNet weights exist only in PyTorch/TF format and can't be loaded into a pure-NumPy autograd engine, so every architecture trains from scratch. What's compared are the architectural *patterns* (residual, inception, fire modules, dense connections).
- **Important:** the fixes that made this laptop-safe (vectorized conv/pooling, autograd chain fixes) must be **pushed to GitHub** — the notebook clones the remote, not your local working tree. Commit & push the `nn/` and `cnn_benchmark/` changes first, or cells 5/6 will fail on the old code.
- Full-data run is ~2 h on free Colab; use `SUBSET_TRAIN = 60000` for the final table.
- Cells 9–14 are safe to re-run anytime (they only read `results.json` and the trained `models` dict in memory).